# Task

Design an in-memory file system that supports creating files and directories, navigating paths, and basic file operations like move and rename

### Clarifying Questions

Q: What kind of characters are allowed in the file and directory names?

For this problem, you can assume names consist of alphanumeric characters, hyphens, underscores, and dots — no spaces or special characters like slashes. Names are also case-sensitive.

Q: For file operations, besides move and rename, can user delete files?

Yes, delete is in scope — users can delete both files and directories. Note that deleting a non-empty directory should also remove all of its contents recursively.

Q: same operations apply to directories?

Yes, the same operations apply to directories — you can create, delete, list, rename, and move directories, subject to the same validity rules (e.g., you cannot move a directory into one of its own descendants).

Q: A directory can contain both subdirectories and a bunch of files?

Yes, that's correct — a directory can contain both subdirectories and files at the same level.

Q: Can you clarify what is needed for navigating path?

"Navigating a path" means resolving a given Unix-style absolute path string (e.g., /home/user/docs) to the actual file or directory it refers to in the in-memory system. All paths start from the root (/), and your system should be able to locate the correct entry by traversing each component of the path in sequence. You don't need to support relative paths or path shortcuts like . or ...

Q: So given a string (path), the method should return a reference to the actual file or directory?

Yes, that's correct — given an absolute path string, the system should be able to resolve and return a reference to the corresponding file or directory entry in memory.

Q: what happens when user tries to name a file or directory with a name that already exists in the same directory?

That's a name collision — and yes, it should be treated as an invalid operation. If a file or directory already exists with the same name in the same parent directory, the operation should fail and throw a specific exception.

Q: Is there a limit on the length of names?

For this problem, you can assume there's no enforced limit on name length — don't worry about that constraint.

Q: Do I need to support bulk moving of files and directories?

No, bulk operations are out of scope — each move operation applies to a single file or directory at a time.

Q: how about bulk delete?

Bulk delete is out of scope — each delete operation applies to a single file or directory at a time.

Q: Do I need to create actual file or directory on disk?

No — this is a purely in-memory file system. Nothing should be written to or read from disk; all data exists only in memory for the lifetime of the program.

Q: How about UI and logging?

Both are out of scope for this problem — no UI is needed, and you don't need to implement any logging infrastructure.

Q: is there any limit on the number of files and subdirectories a directory can contain?

No enforced limit — a directory can contain any number of files and subdirectories. You can assume memory is sufficient to handle tens of thousands of entries across the entire file system.

Q: if user tries to navigate to a non exist directory or file, what should be the respond?

If a user tries to navigate to a path that doesn't exist, the system should throw a specific exception indicating that the path was not found.

Q: since there is no UI, when user wants to perform an operation, is she going to input the path of the file or directory for the operation?

Yes — since there's no UI, you can think of the public API of your file system as a set of methods that accept path strings as parameters. For example, an operation like delete or move would take the absolute path(s) of the target entry as input.

Q: Names are case sensitive, so two files with the same name but different cases are allowed?

Yes, that's correct — since names are case-sensitive, file.txt and File.txt would be treated as two distinct entries and could coexist in the same directory without conflict.

Q: If name collision happens during a move operation, what should be expected?

If a name collision occurs during a move operation — meaning an entry with the same name already exists in the destination directory — the operation should fail and throw a specific exception, just like it would for any other name collision.

Q: Does calling list() on a folder return a list of names for the files and subfolders within it or the actual objects?

For this problem, list() can return just the names — a list of strings representing the names of the direct children (files and subdirectories) within that directory.

Q: In that case, how to distinguish a file from a folder? Can both have the same name?

Good question — within the same directory, a file and a folder cannot share the same name, since names must be unique among all direct children regardless of type. As for distinguishing them in the list() output, that's a design decision for you to make.

# 1. Requirements

### My Answer - 1st attempt
1. An in-memory file system that contains files and directories
2. User can create files or directories at the root level or under any existing directories.
3. A directory can contain both files and subdirectories at the same level. No limit on the
    number of files or subdirectories.
4. File and directory names can contain alphanumeric characters, hyphens, underscores, and dots. 
    All other characters are invalid. Names are case sensitive. No limit on name length.
5. User can perform these actions on a file or directory: navigate, rename, move or delete.
    All operations accept path strings as input.
6. User can also list the content of a directory.
7. Deleting a non-empty directory also removed all of its contents recursively.
8. Raise a specific exception for any of these error conditions:
    - invalid names when creating or renaming a file or directory
    - name collision when creating, renaming or moving a file or directory
    - perform actions on non-exist files or directories
    - moving a directory to its own descendants
    - moving a file or directory to a destination that is a file (not directory) 
    - list operation on a file


Out of Scope:
- Bulk operations
- Support relative paths or path shortcuts
- Files or directories written or read from disk
- UI or logging

### Feedback

What went well

You covered all the primary operations clearly — create, delete, list, rename, move, and path resolution — and correctly noted that all operations accept path strings as input, which shows you internalized the API design well. 
Your error handling section is genuinely strong. You caught name collisions across multiple operations, moving a directory into its own descendant, navigating to non-existent paths, and even the edge case of trying to list a file — that last one especially shows sharp thinking. 
Your out-of-scope section is well-considered and keeps the design tightly focused on what was actually discussed. 

What needs improvement

One error case worth adding is what happens when a user tries to delete the root directory — that's a meaningful invalid operation that your system should handle with a specific exception, and it's easy to overlook until you're mid-implementation. 
You mentioned that creating files or directories works 'at the root level or under any existing directories,' but it would be worth explicitly calling out that creating an entry at a path where the parent directory doesn't exist should also throw a specific exception — right now your error handling section doesn't cover that case. 

Minor suggestions

You might consider briefly noting that files store string content — it's a small detail, but it helps clarify the data model when you move into class design and someone asks what a file actually holds. 

### My Answer - 2nd attempt
1. An in-memory file system that contains files and directories
2. User can create files or directories at the root level or under any existing directories.
3. A directory can contain both files and subdirectories at the same level. No limit on the
    number of files or subdirectories.
4. A file can store string content.    
5. File and directory names can contain alphanumeric characters, hyphens, underscores, and dots. 
    All other characters are invalid. Names are case sensitive. No limit on name length.
6. User can perform these actions on a file or directory: navigate, rename, move or delete.
    All operations accept path strings as input.
7. User can also list the content of a directory.
8. Deleting a non-empty directory also removed all of its contents recursively.
9. Raise a specific exception for any of these error conditions:
    - invalid names when creating or renaming a file or directory
    - name collision when creating, renaming or moving a file or directory
    - creating a file or directory under a parent directory that does not exist  
    - perform rename, move, or delete on non-exist files or directories
    - moving a directory to its own descendants
    - moving a file or directory to a destination that is a file (not directory) 
    - moving a file or directory to a non-exist destination
    - list operation on a file
    - list operation on a non-exist directory
    - deleting, moving, or renaming the root directory
    

Out of Scope:
- Bulk operations
- Support relative paths or path shortcuts
- Files or directories written or read from disk
- UI or logging

### Feedback

What went well

You addressed both pieces of feedback from last time: you added the root directory protection ('deleting, moving, or renaming the root directory') and explicitly called out that creating an entry where the parent doesn't exist throws an exception — those are exactly the gaps we discussed, and it's great to see them covered cleanly. 

Your error handling section is remarkably thorough. You caught name collisions across create, rename, and move; moving a directory into its own descendants; listing a file instead of a directory; and navigating to non-existent paths — these edge cases often trip people up mid-implementation, so having them written down now is a real asset. 

Your out-of-scope section is tight and well-reasoned — bulk operations, relative paths, persistence, and UI are all correctly excluded, keeping the design focused on what actually matters here. 


### Sample Answer

1. Hierarchical file system with single root directory
2. Files store string content
3. Folders contain files and other folders
4. Create and delete files and folders
5. List contents of a folder
6. Navigate/resolve absolute paths (e.g., /home/user/docs)
7. Rename and move files and folders
8. Retrieve full path from any file/folder reference
9. Scale to tens of thousands of entries in memory

Out of Scope:
- Search functionality
- Relative path resolution (../ or ./)
- Permissions, ownership, timestamps
- File type-specific behavior
- Persistence / disk storage
- Symbolic links
- UI layer

# 2. Entities and Relationships

### My Answer - Entities
- FileSystem (FileManager)
- File
- Directory

### Feedback

What went well

You nailed all three core entities — FileSystem, File, and Directory cover exactly what this system needs to model. 
Nice touch including FileManager as an alternate name for FileSystem — shows you're already thinking about how this thing gets used as an API entry point. 

### My Answer - Relationships (1st attempt)

FileManager is the entry point and orchestrator of the system. It contains a reference to the root directory. 
It also owns validation and enforces constraints and rules. All operations (create, rename, delete, list, navigate) have to be through this class.

File is a data holder for properties like name, and content.

Directory is a data holder for name and list of files and subdirectories within it.

### Feedback (1st attempt)

What went well

You clearly identified FileManager as the orchestrator — explaining that all operations flow through it and that it enforces constraints is exactly the right framing. 
You showed the key ownership relationships well: FileManager owns the root directory, Directory holds a list of files and subdirectories, and File holds its content. 

Minor suggestions

One thing worth thinking about: since both File and Directory share common properties like 'name', it could be useful to mention whether they share a common base type or interface — that would help clarify how Directory's children list can hold both files and subdirectories together. 

### My Answer - Relationships (2nd attempt)

FileManager is the entry point and orchestrator of the system. It contains a reference to the root directory. 
It also owns validation and enforces constraints and rules. All operations (create, rename, delete, list, navigate) have to be through this class.

File is a data holder for properties like name, and content.

Directory is a data holder for name and list of files and subdirectories within it.

The File and Directory class can share the same base class or interface, so that the children list in the Directory class can hold objects from both classes. 

### Feedback (2nd attempt)

What went well

You clearly identified FileManager as the orchestrator — explaining that all operations flow through it and that it enforces validation and constraints is exactly the right framing for a facade/entry-point pattern. 

You addressed the previously flagged gap directly: your explanation that File and Directory can share a base class or interface so that the Directory's children list can hold both types shows solid understanding of how the composite structure fits together. 

The ownership chain is well laid out — FileManager holds the root Directory, Directory holds a list of files and subdirectories, and File holds its own content and name. 

### Sample Answer

"FileSystem is the Facade - it owns the root Folder and provides the public API for all path-based operations. File and Folder both inherit from FileSystemEntry, sharing name, parent pointer, and getPath() behavior (Composite pattern). Each Folder has a Map of children keyed by name, containing any mix of Files and other Folders. Each entry stores a parent pointer to its containing Folder, enabling dynamic path computation by walking up the tree. Files are leaf nodes with content, Folders are container nodes with children."

# 3. Class Design

The FileSystem class has an internal reference to the root folder. It offers public APIs for all file and folder operations.

The design uses composite pattern. Internal objects like File and Folder implement the abstract class FileSystemItem.
 
The APIs return FileSystemMetadata which are implemented as immutable objects for Files or Folders. This way, users cannot modify the internal objects directly. This is to adhere to the requirement that the FileSystem class is the single entry point for all operations.  

 

```
ENUM ItemType:
    FILE
    FOLDER

class FileSystem:
    - _root: Folder
    + FileSystem()
    + creatFile(path, content) -> FileMetadata
    + createFolder(path) -> FolderMetadata
    + delete(path)
    + _getParentAndChild(path) -> Folder, FileSystemItem
    + get(path) -> FileSystemMetadata
    + _isNameValid(name) -> bool
    + list(path) -> List<FileSystemMetadata>
    + move(sourcePath, destinationPath)
    + rename(newName, path)

abstract class FileSystemItem:
    - _name: string
    + getName() -> string
    + setName(name)
    + getMetadata() -> FileSystemMetadata
    + isDirectory() -> bool

class File implement FileSystemItem:
    - _content: string?
    + File(name, content)
    + getContent() -> string
    + setContent(content)
    
class Folder implement FileSystemItem:
    - _children: Map<string, FileSystemItem>
    + Folder(name)
    + addChild(FileSystemItem)
    + getChild(name) -> FileSystemItem
    + getChildren() -> List<FileSystemItem>
    + hasChild(name) -> bool
    + removeChild(name)
    
abstract class FileSystemMetadata:
    - name: string
    - type: ItemType
    + FileSystemMetadata(name, itemType)
    + getName() -> string
    + getItemType() -> ItemType

class FileMetadata implement FileSystemMetadata:
    - _content: string?
    + FileMetadata(name, content)
    + getContent() -> string
    
class FolderMetadata implement FileSystemMetadata:
    - _childCount: int
    + FolderMetadata(name, childCount)
    + getChildCount() -> int

```    


### Feedback
What went well

You nailed the Composite pattern with FileSystemItem as the shared abstraction for File and Folder, and your verbal explanation showed you understand exactly why that structure works here. 
Your Folder using a Map<string, FileSystemItem> for _children is exactly the right call — O(1) lookups will matter at scale, and you clearly thought about that. 
The FileSystem class is cleanly designed as a facade with all the right public operations: createFile, createFolder, delete, list, move, rename, and get. That's complete coverage of the required behaviors. 
What needs improvement

Your FileSystemItem (and by extension File and Folder) is missing a parent pointer back to the containing Folder. Without it, computing a full path like /home/user/docs requires the FileSystem to track parent relationships externally — but with a parent pointer, any entry can reconstruct its own path by walking up the tree, which makes move and rename much cleaner to implement. 
Minor suggestions

Since Folder.addChild() and removeChild() will need to maintain the bidirectional link (setting and clearing the parent pointer on the child), it's worth thinking now about how those two methods will coordinate that — it's easy to forget the cleanup side in removeChild. 


### Sample Answer

"This design uses the Composite pattern (FileSystemEntry as shared base, File as leaf, Folder as composite) and the Facade pattern (FileSystem as the single entry point). Path resolution and tree navigation are encapsulated in FileSystem. Parent pointers enable O(depth) path computation without cascading updates on rename/move. Folder uses a Map for O(1) child lookups."

```
abstract class FileSystemEntry:
    - name: string
    - parent: Folder?

    + FileSystemEntry(name)
    + getName() -> string
    + setName(name)
    + getParent() -> Folder?
    + setParent(Folder?)
    + getPath() -> string
    + isDirectory() -> boolean  // abstract


class File extends FileSystemEntry:
    - content: string

    + File(name, content)
    + getContent() -> string
    + setContent(content)
    + isDirectory() -> false


class Folder extends FileSystemEntry:
    - children: Map<string, FileSystemEntry>

    + Folder(name)
    + isDirectory() -> true
    + addChild(entry) -> boolean
    + removeChild(name) -> FileSystemEntry?
    + getChild(name) -> FileSystemEntry?
    + hasChild(name) -> boolean
    + getChildren() -> List<FileSystemEntry>


class FileSystem:
    - root: Folder

    + FileSystem()
    + createFile(path, content) -> File
    + createFolder(path) -> Folder
    + delete(path)
    + list(path) -> List<FileSystemEntry>
    + get(path) -> FileSystemEntry
    + rename(path, newName)
    + move(srcPath, destPath)

```